# 20 Data Analysis Automation — Real-World Automation Without AI

This notebook is a practical, executable lesson. Each step follows **Context → Source Problem → Detect → Fix → Verify → Business Result → Exercise**.

## 1. Automation boundary

Automate repetitive data analyst work: extract, validate, query, calculate KPIs, summarize, publish and schedule. This phase does not require AI.

## 2. Automated CSV extraction

Detect the newest source file and process it.

In [ ]:
from pathlib import Path
files=sorted(Path('data/incoming').glob('*.csv'),key=lambda p:p.stat().st_mtime,reverse=True)
print('latest:',files[0] if files else 'No files')

## 3. Automated database extraction

Connect to SQLite and extract only the required rows.

In [ ]:
import sqlite3,pandas as pd
con=sqlite3.connect('../05_data_sources/source_data/sqlite/sales_source.db')
df=pd.read_sql_query("SELECT * FROM sales_transactions WHERE sales_amount > 0",con)
print(df.shape)

## 4. Automated quality gate

Stop the pipeline when a critical rule fails.

In [ ]:
def quality_gate(df):
    checks={
      'transaction_id_unique': df.transaction_id.is_unique,
      'positive_sales': df.sales_amount.ge(0).all(),
      'positive_quantity': df.quantity.gt(0).all()
    }
    print(checks)
    if not all(checks.values()): raise ValueError('Quality gate failed')
quality_gate(df)

## 5. Automated KPI calculation

Generate the same management KPIs every day/week/month.

In [ ]:
kpi={
 'sales':df.sales_amount.sum(),
 'units':df.quantity.sum(),
 'transactions':df.transaction_id.nunique(),
 'profit':df.profit.sum(),
 'aov':df.sales_amount.sum()/df.transaction_id.nunique()
}
print(kpi)

## 6. Automated summary text

Generate a deterministic management summary from calculated numbers.

In [ ]:
summary=f"Sales were {kpi['sales']:,.0f} across {kpi['transactions']:,} transactions. Units sold were {kpi['units']:,} and profit was {kpi['profit']:,.0f}. Average order value was {kpi['aov']:,.2f}."
print(summary)

## 7. Schedule with Python/cron

The script can be called by cron, Windows Task Scheduler or another scheduler.

In [ ]:
print('Example cron: 0 19 * * * python /path/to/run_daily_analysis.py')

## Exercises

1. Run every code cell and inspect the output.
2. Modify at least one input value to create a data problem.
3. Detect the problem with code.
4. Fix it without hiding the original record.
5. Verify the result and explain the business impact.